# Ticket 24 hidden-test delivery v1

This delivery-only notebook runs the exact frozen Ticket 11 candidate on the competition hidden-test input supplied by Kaggle at scoring time. It loads the immutable candidate and checkpoints, the separately hashed Kaggle adapter, this request-bound notebook, and the pinned offline runtime wheels. A separate owner-approval input must authorize this exact Ticket 24 request before inference. The competition test files are supplied by Kaggle and are not checksum-pinned because their hidden version is selected at scoring time. The notebook writes `submission.csv` and execution evidence to the working directory.

Internet is disabled. Reports are not an inference input. The adapter verifies the request, exact owner approval, notebook hash, adapter hash, pinned candidate environment, and official competition mount before inference. Candidate weights, preprocessing, and inference behavior remain bound to the frozen Ticket 11 candidate. A completed run will provide hidden-test execution evidence; it does not itself claim platform acceptance or final selection.

In [ ]:
import hashlib
import importlib
import importlib.metadata as importlib_metadata
import json
import os
import platform
import re
import subprocess
import sys
import time
from pathlib import Path

notebook_started_at = time.perf_counter()
notebook_cpu_started = os.times()
startup_setup_started = notebook_started_at
EXPECTED_REQUEST_ID = 'ticket-24-ticket-11-local-candidate-20260923-hidden-v1'
EXPECTED_CANDIDATE_MANIFEST_CHECKSUM = 'cd4e9d45832844211c6ccb77f794b5241e164a4da5860a67193ea8a4deec95bc'

def _canonical_json(value):
    return json.dumps(value, ensure_ascii=True, allow_nan=False, sort_keys=True, separators=(',', ':'))

def _json_sha256(value):
    return hashlib.sha256(_canonical_json(value).encode('utf-8')).hexdigest()

def _mounted_files(root, filename):
    found = []
    frontier = [(root, 0)]
    payload_directory_names = {'candidate', 'code', 'test_series', 'train', 'test'}
    while frontier:
        directory, depth = frontier.pop(0)
        children = sorted(directory.iterdir(), key=lambda path: str(path))
        found_in_directory = False
        for child in children:
            if child.is_symlink():
                continue
            if child.is_file() and child.name == filename:
                found.append(child)
                found_in_directory = True
        if found_in_directory or depth >= 5:
            continue
        for child in children:
            if child.is_symlink() or not child.is_dir() or child.name in payload_directory_names:
                continue
            frontier.append((child, depth + 1))
    return found

def _locked_pins(path):
    pins = {}
    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        match = re.fullmatch(r'([A-Za-z0-9_.-]+)==(.+)', line)
        if match is None:
            raise RuntimeError(f'unexpected frozen runtime lock line: {line}')
        package = re.sub(r'[-_.]+', '-', match.group(1).casefold())
        pins[package] = match.group(2).strip()
    return pins

def _metadata_versions(packages):
    versions = {}
    for package in sorted(packages):
        try:
            versions[package] = importlib_metadata.version(package)
        except importlib_metadata.PackageNotFoundError:
            versions[package] = None
    return versions

def _runtime_snapshot(packages):
    return {
        'python_version': platform.python_version(),
        'python_pin': None,
        'packages': _metadata_versions(packages),
    }

input_root = Path('/kaggle/input')
request_matches = []
for request_path in _mounted_files(input_root, 'request.json'):
    try:
        request_candidate = json.loads(request_path.read_text(encoding='utf-8'))
        if not isinstance(request_candidate, dict):
            continue
        candidate_reference = request_candidate.get('candidate')
        if not isinstance(candidate_reference, dict):
            continue
        request_checksum = request_candidate.get('request_checksum')
        unsigned_request = dict(request_candidate)
        unsigned_request.pop('request_checksum', None)
    except (OSError, UnicodeError, json.JSONDecodeError, TypeError, ValueError):
        continue
    if (
        request_candidate.get('schema_version') == 'rsna-knee-kaggle-delivery-v1'
        and request_candidate.get('status') == 'owner_approval_required'
        and request_candidate.get('request_id') == EXPECTED_REQUEST_ID
        and candidate_reference.get('manifest_checksum') == EXPECTED_CANDIDATE_MANIFEST_CHECKSUM
        and _json_sha256(unsigned_request) == request_checksum
    ):
        request_matches.append((request_path, request_checksum, request_candidate))
if len(request_matches) != 1:
    raise RuntimeError(f'expected one exact Ticket 24 request mount, found {len(request_matches)}')
bootstrap_request_path, expected_request_checksum, bootstrap_request = request_matches[0]
bootstrap_delivery_root = bootstrap_request_path.parent
bootstrap_candidate_root = bootstrap_delivery_root / 'candidate'
runtime_reference = bootstrap_request.get('delivery_runtime')
if not isinstance(runtime_reference, dict):
    raise RuntimeError('the exact Ticket 24 request has no offline runtime wheelhouse')
wheel_references = runtime_reference.get('offline_wheels')
if not isinstance(wheel_references, list) or not wheel_references or not all(isinstance(item, dict) for item in wheel_references):
    raise RuntimeError('the exact Ticket 24 request has no offline runtime wheelhouse references')
wheel_paths = []
wheel_records = []
offline_wheel_verification_started = time.perf_counter()
for wheel_reference in wheel_references:
    wheel_name = str(wheel_reference.get('filename', '')).strip()
    wheel_relative_path = str(wheel_reference.get('relative_path', '')).strip()
    if not wheel_name or wheel_name in {'.', '..'} or '/' in wheel_name or '\\' in wheel_name:
        raise RuntimeError('the request runtime wheel filename is not path-safe')
    if wheel_relative_path != f'runtime/{wheel_name}':
        raise RuntimeError('the request runtime wheel path is not canonical')
    wheel_path = bootstrap_delivery_root / wheel_relative_path
    if wheel_path.is_symlink() or not wheel_path.is_file():
        raise RuntimeError(f'the requested offline runtime wheel is not mounted: {wheel_name}')
    wheel_sha256 = hashlib.sha256(wheel_path.read_bytes()).hexdigest()
    if wheel_sha256 != wheel_reference.get('sha256'):
        raise RuntimeError(f'the mounted runtime wheel checksum differs from the request: {wheel_name}')
    wheel_paths.append(wheel_path)
    wheel_records.append({
        'filename': wheel_reference['filename'],
        'bytes': wheel_path.stat().st_size,
        'sha256': wheel_reference['sha256'],
        'package': wheel_reference['package'],
        'version': wheel_reference['version'],
        'tags': wheel_reference.get('tags', []),
    })
offline_wheel_verification_seconds = time.perf_counter() - offline_wheel_verification_started

locked_pins = _locked_pins(bootstrap_candidate_root / 'environment' / 'offline-requirements.lock')
candidate_environment = bootstrap_request['candidate']['environment']
candidate_pins = {
    re.sub(r'[-_.]+', '-', str(package).casefold()): str(version)
    for package, version in candidate_environment['runtime_pins'].items()
}
if locked_pins != candidate_pins:
    raise RuntimeError('the mounted runtime lock differs from the candidate runtime pins')
runtime_before = _runtime_snapshot(locked_pins)
runtime_site = Path('/kaggle/working/runtime-site')
runtime_site.mkdir(parents=True, exist_ok=True)
install_command = [
    sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps',
    '--target', str(runtime_site), '--upgrade', '--disable-pip-version-check',
] + [str(path) for path in wheel_paths]
offline_wheel_install_started = time.perf_counter()
try:
    subprocess.run(install_command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
except subprocess.CalledProcessError as error:
    failed_install_evidence = {
        'status': 'install_failed',
        'wheels': wheel_records,
        'installation': {
            'no_index': True,
            'no_deps': True,
            'local_wheel_only': True,
            'internet_enabled': False,
            'isolated_target': str(runtime_site),
        },
        'environment_before': runtime_before,
        'environment_after': None,
        'error': str(error),
    }
    Path('/kaggle/working/runtime-install.json').write_text(json.dumps(failed_install_evidence, indent=2, sort_keys=True), encoding='utf-8')
    raise
finally:
    offline_wheel_install_seconds = time.perf_counter() - offline_wheel_install_started
probe_environment = dict(os.environ)
probe_environment['PYTHONPATH'] = str(runtime_site)
runtime_probe_code = """import importlib.metadata as importlib_metadata
import json
import platform
import numpy
import pydicom
import torch
packages = {package: importlib_metadata.version(package) for package in ('numpy', 'pydicom', 'torch')}
print(json.dumps({'python_version': platform.python_version(), 'python_pin': None, 'packages': packages, 'imported_packages': {'numpy': numpy.__version__, 'pydicom': pydicom.__version__, 'torch': torch.__version__}}, sort_keys=True))
"""
offline_runtime_probe_started = time.perf_counter()
try:
    runtime_after = json.loads(
        subprocess.check_output(
            [sys.executable, '-c', runtime_probe_code],
            env=probe_environment,
            text=True,
        ).strip()
    )
except (subprocess.CalledProcessError, json.JSONDecodeError) as error:
    failed_probe_evidence = {
        'status': 'runtime_probe_failed',
        'wheels': wheel_records,
        'installation': {
            'no_index': True,
            'no_deps': True,
            'local_wheel_only': True,
            'internet_enabled': False,
            'isolated_target': str(runtime_site),
        },
        'environment_before': runtime_before,
        'environment_after': None,
        'error': str(error),
    }
    Path('/kaggle/working/runtime-install.json').write_text(json.dumps(failed_probe_evidence, indent=2, sort_keys=True), encoding='utf-8')
    raise
finally:
    offline_runtime_probe_seconds = time.perf_counter() - offline_runtime_probe_started
for package, expected_version in locked_pins.items():
    if runtime_after['packages'].get(package) != expected_version:
        raise RuntimeError(f'runtime package {package} did not reach its frozen pin')
    if runtime_after['imported_packages'].get(package) != expected_version:
        raise RuntimeError(f'imported runtime package {package} did not reach its frozen pin')
runtime_install_evidence = {
    'status': 'installed_offline',
    'wheels': wheel_records,
    'installation': {
        'no_index': True,
        'no_deps': True,
        'local_wheel_only': True,
        'internet_enabled': False,
    },
    'environment_before': runtime_before,
    'environment_after': runtime_after,
}
Path('/kaggle/working/runtime-install.json').write_text(json.dumps(runtime_install_evidence, indent=2, sort_keys=True), encoding='utf-8')
startup_setup_seconds = time.perf_counter() - startup_setup_started
notebook_cpu_finished = os.times()
notebook_cpu_user_seconds = notebook_cpu_finished.user - notebook_cpu_started.user
notebook_cpu_system_seconds = notebook_cpu_finished.system - notebook_cpu_started.system
runtime_config_path = Path('/kaggle/working/ticket24-runtime-config.json')
runtime_config_path.write_text(json.dumps({
    'input_root': str(input_root),
    'expected_request_checksum': expected_request_checksum,
    'expected_candidate_manifest_checksum': EXPECTED_CANDIDATE_MANIFEST_CHECKSUM,
    'input_slug': bootstrap_request['execution_input']['mount_slug'],
    'runtime_site': str(runtime_site),
    'code_root': str(bootstrap_delivery_root / 'code'),
    'runtime_install_path': '/kaggle/working/runtime-install.json',
    'output_path': '/kaggle/working/submission.csv',
    'evidence_path': '/kaggle/working/kaggle-execution.json',
    'private_evidence_path': '/kaggle/working/private-execution.json',
    'notebook_started_at': notebook_started_at,
    'startup_setup_seconds': startup_setup_seconds,
    'notebook_timing_details': {
        'notebook_startup_preflight_seconds': startup_setup_seconds,
        'offline_wheel_verification_seconds': offline_wheel_verification_seconds,
        'offline_wheel_install_seconds': offline_wheel_install_seconds,
        'offline_runtime_probe_seconds': offline_runtime_probe_seconds,
        'notebook_cpu_user_seconds': notebook_cpu_user_seconds,
        'notebook_cpu_system_seconds': notebook_cpu_system_seconds,
    },
}, indent=2, sort_keys=True), encoding='utf-8')
inference_script = Path('/kaggle/working/run-ticket24-inference.py')
inference_script.write_text("""import json
import sys
import time
from pathlib import Path

config = json.loads(Path('/kaggle/working/ticket24-runtime-config.json').read_text(encoding='utf-8'))
sys.path.insert(0, config['runtime_site'])
sys.path.insert(0, config['code_root'])
runtime_config = Path(config['runtime_site'])
delivery_code = Path('/kaggle/input')
imports_started = time.perf_counter()
import torch
from rsna_knee.kaggle import discover_kaggle_inputs, run_kaggle_baseline
imports_seconds = time.perf_counter() - imports_started
input_discovery_started = time.perf_counter()
layout = discover_kaggle_inputs(
    config['input_root'],
    expected_request_checksum=config['expected_request_checksum'],
    expected_candidate_manifest_checksum=config['expected_candidate_manifest_checksum'],
    competition_slug=config['input_slug'],
)
input_discovery_seconds = time.perf_counter() - input_discovery_started
request = json.loads(layout.request_path.read_text(encoding='utf-8'))
runtime_install_evidence = json.loads(Path(config['runtime_install_path']).read_text(encoding='utf-8'))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
result = run_kaggle_baseline(
    layout.candidate_root,
    layout.competition_root,
    Path(config['output_path']),
    evidence_path=Path(config['evidence_path']),
    private_evidence_path=Path(config['private_evidence_path']),
    delivery_request_checksum=request['request_checksum'],
    delivery_notebook_sha256=request['delivery_notebook']['sha256'],
    delivery_adapter_sha256=request['delivery_adapter']['sha256'],
    delivery_request_root=layout.delivery_root,
    owner_approval_path=layout.approval_path,
    delivery_notebook_path=layout.delivery_notebook_path,
    delivery_adapter_path=layout.delivery_adapter_path,
    delivery_runtime_wheel_paths=layout.runtime_wheel_paths,
    runtime_install_evidence=runtime_install_evidence,
    notebook_stage_timings={'startup_setup': config['startup_setup_seconds'], 'imports': imports_seconds},
    notebook_timing_details={**config['notebook_timing_details'], 'input_discovery_seconds': input_discovery_seconds, 'child_import_seconds': imports_seconds},
    validated_layout=layout,
    device=device,
    notebook_started_at=config['notebook_started_at'],
)
print(json.dumps(result.evidence, indent=2, sort_keys=True))
""", encoding='utf-8')


In [ ]:
child_result = subprocess.run([sys.executable, str(inference_script)], check=False, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
if child_result.stdout:
    print(child_result.stdout)
if child_result.stderr:
    print(child_result.stderr)
if child_result.returncode != 0:
    raise subprocess.CalledProcessError(child_result.returncode, child_result.args)


## Recovery evidence

This delivery does not synthesize recovery evidence from a completed output CSV. The adapter records inference-level save/reload and interruption recovery as unverified unless those behaviors are demonstrated separately. A successful hidden-test submission does not by itself pass the recovery gate.

## After the run

Keep the private runtime-install audit, execution file, and `submission.csv` private. The execution JSON records the exact candidate, adapter, notebook, offline wheel checksums, before/after runtime pins, resource measurements, workload, stage timings, projection, output checksum, and fallback accounting. This run does not submit to the leaderboard; platform acceptance remains pending. Unknown hidden fallback information must remain unknown.